# 02 Grid and Isochrone Proxy

## Method Role

This notebook documents the grid and travel-mode stage requested in the brief. The ideal target is a 500 m grid with walk, bike, transit, and car 15-minute network isochrones, followed by spatial joins between each isochrone and the POI / AQI layers. The current final package implements a reproducible proxy: POIs are assigned to H3 resolution 8 cells, nearby cells are aggregated with different mode radii, and a linked 500 m square grid proxy is exported from the scored H3 centroids.

## Why a Proxy Is Used Here

The provided `shanghai-network-extract.ipynb` shows the intended graph-tool direction, but it depends on `graph_tool`, `policosm`, and absolute paths from the instructor's machine. The local Python runtime also does not include the geospatial stack needed to read the parquet road network directly. Until that environment is cleaned up, this notebook uses H3 neighborhood rings to support reproducible scoring, Web visualization, district filtering, and exportable recommendations. The proxy should later be replaced by true shortest-path travel-time isochrones using `shanghai-roads-simplified.parquet`.

## Mode Proxy Definition

| Mode | Current reproducible proxy | Intended final replacement |
| --- | --- | --- |
| Walk | H3 neighborhood radius 2 | 15-minute pedestrian network isochrone from each 500 m grid center |
| Bike | H3 neighborhood radius 4 | 15-minute cycling network isochrone with cycling lane / speed assumptions |
| Transit | H3 neighborhood radius 5 and transit POI count | Scheduled transit travel-time catchment |
| Car | H3 neighborhood radius 7, comparison only | Road-network travel-time catchment, not used in baseline score |

Caching strategy: the heavy POI-to-H3 and proxy aggregation outputs are written to `outputs/sh15_trackA_h3_r8_scored.geojson`, `webapp/data/sh15_trackA_h3_r8_scored.geojson`, and `outputs/sh15_trackA_500m_grid_proxy.geojson`. Re-running the build refreshes these files deterministically from the local raw data.

## Output Contract

Running this notebook calls the Node data builder and writes a scored GeoJSON layer to both `outputs/` and `webapp/data/`. The Web app reads the `webapp/data` copy directly.


In [1]:
from pathlib import Path
import os
import subprocess

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
npm = 'npm.cmd' if os.name == 'nt' else 'npm'
subprocess.run([npm, 'run', 'build:data'], cwd=ROOT, check=True)
subprocess.run([npm, 'run', 'build:grid'], cwd=ROOT, check=True)


In [2]:
import json

geojson_path = ROOT / 'outputs' / 'sh15_trackA_h3_r8_scored.geojson'
grid_path = ROOT / 'outputs' / 'sh15_trackA_500m_grid_proxy.geojson'
geojson = json.loads(geojson_path.read_text(encoding='utf-8'))
grid = json.loads(grid_path.read_text(encoding='utf-8'))
first = geojson['features'][0]['properties']
print('H3 features:', len(geojson['features']))
print('H3 metadata:', geojson['metadata'])
print('500m grid proxy features:', len(grid['features']))
print('500m grid metadata:', grid['metadata'])
print('Mode score fields:', [k for k in first if k.endswith('_baseline_score')])


H3 features: 11344
H3 metadata: {'h3_resolution': 8, 'generated_at': '2026-06-24T05:40:53.765Z', 'scoring_method': 'H3 neighborhood proxy accessibility with empirical percentile scaling; true graph isochrones not yet substituted.', 'feature_count': 11344}
500m grid proxy features: 11344
500m grid metadata: {'generated_at': '2026-06-24T05:40:54.539Z', 'grid_spacing_m': 500, 'feature_count': 11344, 'linked_h3_resolution': 8, 'method': '500m square proxy centered on each scored H3 r8 cell centroid; used as reproducible evidence of the requested grid stage while true 4-mode network isochrones remain pending.'}
Mode score fields: ['walk_baseline_score', 'bike_baseline_score', 'transit_baseline_score', 'car_baseline_score']
